##Enterprise Fleet Analytics Pipeline: Focuses on the business outcome (analytics) and the domain (fleet/logistics)

In [0]:
%sql
-- ----------------------------------------
-- Create or Replace Temporary View
-- ----------------------------------------
-- Temporary views exist only for the current Spark session.
-- They are useful for lightweight, session-only queries without persisting data.

CREATE OR REPLACE TEMPORARY VIEW shipment_temp_1
(
  shipment_id INT,     -- Shipment identifier
  first_name STRING,   -- Driver/staff first name
  last_name STRING,    -- Driver/staff last name
  age INT,             -- Age of staff
  role STRING          -- Role (e.g., driver, supervisor)
)
USING CSV
OPTIONS
(
  path   "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt", -- Source CSV file
  header "true",   -- First row contains column names
  inferSchema "false" -- Schema is explicitly defined above, so inference is disabled
);

-- ----------------------------------------
-- Query the temporary view
-- ----------------------------------------
-- This will display all rows from shipment_temp_1
SELECT * FROM shipment_temp_1;

In [0]:
# ----------------------------------------
# 1. Describe schema
# ----------------------------------------
# Instead of using schema, columns, or dtypes, you can use DESCRIBE in SQL
# to get column names and datatypes quickly.
display(spark.sql("DESCRIBE shipment_temp_1"))

# Load the temp view into a DataFrame for PySpark operations
shipment_df1 = spark.sql("SELECT * FROM shipment_temp_1")

# ----------------------------------------
# 2. Understanding duplicates for individual columns
# ----------------------------------------
# Total row count
display(spark.sql("SELECT COUNT(*) FROM shipment_temp_1"))

# Distinct shipment_id count
display(spark.sql("SELECT COUNT(DISTINCT shipment_id) FROM shipment_temp_1"))
display(shipment_df1.dropDuplicates(["shipment_id"]).count())

# Distinct first_name count
display(spark.sql("SELECT COUNT(DISTINCT first_name) FROM shipment_temp_1"))
display(shipment_df1.dropDuplicates(["first_name"]).count())

# Distinct last_name count
display(spark.sql("SELECT COUNT(DISTINCT last_name) FROM shipment_temp_1"))
display(shipment_df1.dropDuplicates(["last_name"]).count())

# ----------------------------------------
# 3. Duplicates for combinations of columns
# ----------------------------------------
# SQL limitation: COUNT(DISTINCT col1, col2, col3) only applies distinct on the first column.
# Correct approach is to use PySpark dropDuplicates with multiple columns.
display(shipment_df1.dropDuplicates(["shipment_id", "first_name", "last_name"]).count())

# ----------------------------------------
# 4. Duplicate rows
# ----------------------------------------
# SQL doesn't support COUNT(DISTINCT *) directly.
# Correct approach: use dropDuplicates() without specifying columns.
display(shipment_df1.dropDuplicates().count())

# ----------------------------------------
# 5. Statistical summary
# ----------------------------------------
# SQL aggregation functions for numeric columns
display(
    spark.sql("""
        SELECT COUNT(shipment_id) AS count,
               AVG(shipment_id) AS avg,
               STDDEV(shipment_id),
               MIN(shipment_id),
               MAX(shipment_id)
        FROM shipment_temp_1
    """)
)

# PySpark summary → gives count, mean, stddev, min, max, percentiles
shipment_df1.summary().show()

In [0]:
# ----------------------------------------
# Create a Spark Session Object
# ----------------------------------------
from pyspark.sql import SparkSession

# Builder pattern: configure and initialize SparkSession
spark = SparkSession.builder.appName("MY Spark Session").getOrCreate()                   # Creates a new session or reuses existing one

# Print the SparkSession object reference
print(spark)

In [0]:
%sql
-- ----------------------------------------
-- Create or Replace Temporary View with Corrupt Record Handling
-- ----------------------------------------
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_2
(
  shipment_id INT,       -- Shipment identifier
  first_name STRING,     -- Staff first name
  last_name STRING,      -- Staff last name
  age INT,               -- Age (read as INT here)
  role STRING,           -- Staff role
  corrupt_record STRING  -- Special column to capture malformed rows
)
USING CSV
OPTIONS
(
  path  "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt", -- Source CSV
  header  "true",              -- First row contains column names
  inferSchema  "false",        -- Schema is explicitly defined above
  columnNameOfCorruptRecord "corrupt_record" -- Capture bad rows here
);

-- ----------------------------------------
-- Identify malformed rows
-- ----------------------------------------
-- Rows with fewer or extra columns will be flagged in corrupt_record
SELECT * FROM shipment_temp_2 WHERE corrupt_record IS NOT NULL;

-- ----------------------------------------
-- Note on non-integer age values
-- ----------------------------------------
-- Since age is defined as INT, Spark will reject non-numeric values.
-- If age were read as STRING, you would need to manually convert text values to integers.
-- Example (conceptual SQL update):
/*
UPDATE person
SET age =
  CASE age
    WHEN 'one' THEN '1'
    WHEN 'two' THEN '2'
    WHEN 'three' THEN '3'
    WHEN 'ten' THEN '10'
    ELSE age
  END;
*/

In [0]:
%sql
-- shipment_temp1
-- Schema: shipment_id, first_name, last_name, age, role
CREATE OR REPLACE TEMPORARY VIEW shipment_temp1
(
  shipment_id INT,
  first_name STRING,
  last_name STRING,
  age INT,
  role STRING
)
USING CSV
OPTIONS (
  path   "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt",
  header "true",
  inferSchema "false"
);

-- shipment_temp2
-- Schema: shipment_id, first_name, last_name, age, role, hub_location, vehicle_type
CREATE OR REPLACE TEMPORARY VIEW shipment_temp2
(
  shipment_id INT,
  first_name STRING,
  last_name STRING,
  age INT,
  role STRING,
  hub_location STRING,
  vehicle_type STRING
)
USING CSV
OPTIONS (
  path   "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source2.txt",
  header "true",
  inferSchema "false"
);

In [0]:
%sql
-- Join shipment_temp1 and shipment_temp2 on shipment_id
SELECT t1.shipment_id
FROM shipment_temp1 t1
JOIN shipment_temp2 t2
  ON t1.shipment_id = t2.shipment_id;

In [0]:
%sql
-- Create catalog only if it doesn't exist
CREATE CATALOG IF NOT EXISTS usecase_data;

-- Create schema only if it doesn't exist
CREATE SCHEMA IF NOT EXISTS usecase_data.logistics_sql_data;

-- Create volume only if it doesn't exist
CREATE VOLUME IF NOT EXISTS usecase_data.logistics_sql_data.projdata;

In [0]:
%sql
-- Create shipment_temp_1 with schema (all STRING for flexibility)
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_1 (
  shipment_id STRING,
  first_name STRING,
  last_name STRING,
  age STRING,
  role STRING,
  hub_location STRING,
  vehicle_type STRING
)
USING CSV
OPTIONS (
  path "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt",
  header "true",
  inferSchema "false"
);

-- Create shipment_temp_2 with same schema
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_2 (
  shipment_id STRING,
  first_name STRING,
  last_name STRING,
  age STRING,
  role STRING,
  hub_location STRING,
  vehicle_type STRING
)
USING CSV
OPTIONS (
  path "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source2.txt",
  header "true",
  inferSchema "false"
);

-- Union both sources into a raw consolidated view
-- Add a column 'data_source' to track origin
CREATE OR REPLACE TEMP VIEW shipment_temp_raw AS
SELECT *, 'source1' AS data_source FROM shipment_temp_1
UNION ALL
SELECT *, 'source2' AS data_source FROM shipment_temp_2;

-- Drop existing permanent table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.shipment_raw_table;

-- Create permanent table from the raw view
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.shipment_raw_table AS
SELECT * FROM shipment_temp_raw;

-- Inspect the consolidated raw data
SELECT * FROM shipment_temp_raw;

In [0]:
%sql
-- Create a temporary view from the Master City List CSV
CREATE OR REPLACE TEMPORARY VIEW master_city_list1 (
  city_name STRING,
  country STRING,
  latitude DECIMAL(10,6),
  longitude DECIMAL(10,6)
)
USING CSV
OPTIONS (
  header "True",   -- First row contains column names
  path "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/Master_City_List.csv",
  inferSchema "False" -- Schema explicitly defined above
);

-- Drop existing permanent table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.master_city_table;

-- Create permanent table from the temporary view
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.master_city_table AS
SELECT * FROM master_city_list1;


SELECT * FROM master_city_list1;

In [0]:
%sql
-- Create a cleaned temporary view from shipment_temp_raw
CREATE OR REPLACE TEMP VIEW shipment_temp_raw1 AS
SELECT
    shipment_id,
    first_name,
    last_name,
    -- Handle invalid or missing age values
    CASE 
        WHEN COALESCE(age, '-1') = 'ten' OR COALESCE(age, '-1') = '' 
        THEN '-1' 
        ELSE age 
    END AS age,
    
    role,
    
    -- Replace placeholder 'Additionalcolumn' with NULL
    CASE 
        WHEN hub_location = 'Additionalcolumn' 
        THEN NULL 
        ELSE hub_location 
    END AS hub_location,
    
    -- Default missing vehicle_type to 'UNKNOWN'
    COALESCE(vehicle_type, 'UNKNOWN') AS vehicle_type,
    
    data_source
FROM shipment_temp_raw
WHERE shipment_id IS NOT NULL
  AND role IS NOT NULL
  AND (first_name IS NOT NULL OR last_name IS NOT NULL);

In [0]:
%sql
-- Create a temporary view from JSON shipment details
CREATE OR REPLACE TEMPORARY VIEW logistics_shipment_temp (
  shipment_id INT,
  order_id STRING,
  source_city STRING,
  destination_city STRING,
  shipment_status STRING,
  cargo_type STRING,
  vehicle_type STRING,
  payment_mode STRING,
  shipment_weight_kg FLOAT,
  shipment_cost FLOAT,
  shipment_date STRING
)
USING JSON
OPTIONS (
  path "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_shipment_detail_3000.json",
  multiLine "True"   -- JSON file contains multi-line records
);

-- Drop existing permanent table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.logistics_shipment_raw_table;

-- Create permanent table from the temporary view
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.logistics_shipment_raw_table AS
SELECT * FROM logistics_shipment_temp;

In [0]:
%sql
-- Create a curated temporary view from the raw shipment data
CREATE OR REPLACE TEMPORARY VIEW logistics_shipment_temp_1 AS
SELECT
    shipment_id,
    order_id,
    source_city,
    destination_city,
    shipment_status,
    cargo_type,
    
    -- Normalize vehicle type to uppercase for consistency
    UPPER(vehicle_type) AS vehicle_type,
    
    payment_mode,
    
    -- Round numeric values to 2 decimal places
    ROUND(CAST(shipment_weight_kg AS DOUBLE), 2) AS shipment_weight_kg,
    ROUND(CAST(shipment_cost AS DOUBLE), 2) AS shipment_cost,
    
    -- Convert shipment_date string into proper DATE type
    TO_DATE(shipment_date, 'yy-MM-dd') AS shipment_date,
    
    -- Add enrichment columns
    'Logistics' AS domain,                 -- Domain tagging
    CURRENT_TIMESTAMP() AS ingestion_timestamp, -- Audit timestamp
    FALSE AS is_expedited                  -- Default flag for expedited shipments
FROM logistics_shipment_temp;

-- Inspect the curated view
-- SELECT * FROM logistics_shipment_temp_1;

In [0]:
%sql
-- Create a curated temporary view from shipment_temp_raw1
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_raw2 AS
SELECT
    -- Cast shipment_id to INT for consistency
    CAST(shipment_id AS INT) AS shipment_id,
    
    -- Rename staff columns for clarity
    first_name AS staff_first_name,
    last_name AS staff_last_name,
    
    -- Cast age to INT
    CAST(age AS INT) AS age,
    
    -- Normalize role to lowercase
    LOWER(role) AS role,
    
    -- Standardize hub city names (capitalize first letter of each word)
    INITCAP(hub_location) AS origin_hub_city,
    
    vehicle_type,
    data_source
FROM shipment_temp_raw1
WHERE shipment_id != 'ten';  -- Filter out invalid shipment_id values

-- Inspect the curated view
SELECT * FROM shipment_temp_raw2;

In [0]:
%sql
-- Row-level distinct: removes exact duplicate rows
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_raw3 AS
SELECT DISTINCT * 
FROM shipment_temp_raw2;

-- Column-level distinct: keeps only one record per shipment_id
-- QUALIFY allows filtering based on window functions
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_raw4 AS
SELECT *
FROM shipment_temp_raw3
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY shipment_id 
    ORDER BY data_source ASC
) = 1;

In [0]:
%sql
-- Create a refined temporary view with standardized columns
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_raw5 AS
SELECT
    shipment_id,
    
    -- Build a full name by concatenating first and last names
    -- TRIM removes extra spaces, CONCAT_WS handles nulls gracefully
    TRIM(CONCAT_WS('', staff_first_name, ' ', staff_last_name)) AS full_name,
    
    age,
    role,
    
    -- Hub city name standardized from earlier step
    origin_hub_city,
    
    vehicle_type,
    data_source,
    
    -- Add load timestamp for auditability
    CURRENT_TIMESTAMP() AS load_dt
FROM shipment_temp_raw4;

-- Inspect the refined dataset
SELECT * FROM shipment_temp_raw5;

In [0]:
%sql
-- Create a feature-rich temporary view from curated shipment data
CREATE OR REPLACE TEMPORARY VIEW logistics_shipment_temp_2 AS
SELECT
    shipment_id,
    order_id,
    source_city,
    destination_city,
    shipment_status,
    cargo_type,
    vehicle_type,
    payment_mode,
    shipment_weight_kg,
    shipment_cost,
    shipment_date,
    domain,
    ingestion_timestamp,
    
    -- Flag expedited shipments based on status
    CASE WHEN shipment_status IN ('IN_TRANSIT','DELIVERED') THEN TRUE ELSE FALSE END AS is_expedited,
    
    -- Route identifiers
    CONCAT_WS('', source_city, '-', destination_city) AS route_segment,
    CONCAT_WS('', vehicle_type, '_', shipment_id) AS vehicle_identifier,
    
    -- Date breakdown
    YEAR(shipment_date) AS shipment_year,
    MONTH(shipment_date) AS shipment_month,
    CASE WHEN DAYOFWEEK(shipment_date) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
    
    -- Cost efficiency
    ROUND(TRY_DIVIDE(shipment_cost, shipment_weight_kg), 2) AS cost_per_kg,
    
    -- Recency
    DATEDIFF(CURRENT_DATE(), shipment_date) AS days_since_shipment,
    
    -- Tax calculation (18% GST assumption)
    ROUND(shipment_cost * 0.18, 2) AS tax_amount,
    
    -- Order parsing
    SUBSTRING(order_id, 1, 3) AS order_prefix,
    SUBSTRING(order_id, 4, LEN(order_id)) AS order_sequence,
    
    -- Shipment date components
    YEAR(shipment_date) AS ship_year,
    MONTH(shipment_date) AS ship_month,
    DAY(shipment_date) AS ship_day,
    
    -- Alternate route representation
    CONCAT_WS('', source_city, '->', destination_city) AS route_lane,
    
    -- High-value shipment flag
    CASE WHEN shipment_cost > 50000 THEN TRUE ELSE FALSE END AS is_high_value
FROM logistics_shipment_temp_1;

In [0]:
%sql
-- Create a masked and enriched temporary view
CREATE OR REPLACE TEMPORARY VIEW shipment_temp_raw6 AS
SELECT
    shipment_id,
    
    -- Mask staff full name: keep first 2 chars and last char, replace middle with '*'
    CONCAT(
        SUBSTRING(full_name, 1, 2),
        REPEAT('*', LEN(full_name) - 3),
        SUBSTRING(full_name, -1)
    ) AS staff_full_name,
    
    age,
    role,
    origin_hub_city,
    vehicle_type,
    data_source,
    
    -- Add load timestamp for audit
    CURRENT_TIMESTAMP() AS load_dt,
    
    -- Projected bonus logic based on role and age
    CASE 
        WHEN role = 'driver' AND age > 50 THEN 0.15
        WHEN role = 'driver' AND age < 30 THEN 0.05
        ELSE 0
    END AS projected_bonus
FROM shipment_temp_raw5;

In [0]:
%sql
-- 1. Driver App Data
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.driver_app_data;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.driver_app_data AS
SELECT staff_full_name, role, origin_hub_city
FROM shipment_temp_raw6;


-- 2. Active Operational Problem Data
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.active_operational_problem_data;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.active_operational_problem_data AS
SELECT *
FROM logistics_shipment_temp_2
WHERE shipment_status IN ('DELAYED','RETURNED');


-- 3. Senior Insurance Audit
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.senior_insurance_audit;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.senior_insurance_audit AS
SELECT *
FROM shipment_temp_raw6
WHERE age > 50;

In [0]:
%sql
-- Create a polished temporary view with formatted attributes
CREATE OR REPLACE TEMPORARY VIEW logistics_shipment_temp_3 AS
SELECT
    shipment_id AS log_shipment_id,
    order_id,
    
    -- Normalize source city names to uppercase
    UPPER(source_city) AS source_city,
    
    destination_city,
    shipment_status,
    cargo_type,
    
    -- Rename vehicle_type for clarity
    vehicle_type AS shipment_vehicle_type,
    
    payment_mode,
    shipment_weight_kg,
    shipment_cost,
    
    -- Add formatted cost in INR with currency symbol
    CONCAT('₹', CAST(shipment_cost AS STRING)) AS shipment_cost_inr,
    
    shipment_date,
    domain,
    ingestion_timestamp,
    is_expedited,
    route_segment,
    vehicle_identifier,
    shipment_year,
    shipment_month,
    is_weekend,
    cost_per_kg,
    days_since_shipment,
    tax_amount,
    order_prefix,
    order_sequence,
    ship_year,
    ship_month,
    ship_day,
    route_lane,
    is_high_value
FROM logistics_shipment_temp_2;

In [0]:
%sql
-- 1. Regional Staffing Analysis
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.regional_staffing_analysis;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.regional_staffing_analysis AS
SELECT 
    origin_hub_city, 
    COUNT(*) AS staff_count
FROM shipment_temp_raw6
GROUP BY origin_hub_city;


-- 2. Fleet Capacity Analysis
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.fleet_capacity_analysis;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.fleet_capacity_analysis AS
SELECT 
    shipment_vehicle_type, 
    SUM(shipment_weight_kg) AS total_weight
FROM logistics_shipment_temp_3
GROUP BY shipment_vehicle_type;

In [0]:
%sql
-- Drop existing table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.daily_dispatch_schedule;

-- Create a new table with prioritized dispatch order
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.daily_dispatch_schedule AS
SELECT
    *
FROM logistics_shipment_temp_3
ORDER BY shipment_date ASC,   -- Earliest shipments first
         shipment_cost DESC;  -- Within the same date, higher-cost shipments prioritized

In [0]:
%sql
-- Drop existing table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.critical_delays;

-- Create a new table capturing the top 10 delayed shipments
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.critical_delays AS
SELECT *
FROM logistics_shipment_temp_3
WHERE shipment_status = 'DELAYED'
ORDER BY shipment_date ASC,   -- Earliest delayed shipments first
         shipment_cost DESC   -- Within the same date, prioritize higher-cost shipments
LIMIT 10;                     -- Restrict to top 10 records

In [0]:
%sql
-- 1. Staff with Shipments
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.staff_with_shipments;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.staff_with_shipments AS
SELECT *
FROM shipment_temp_raw6 t1
INNER JOIN logistics_shipment_temp_3 t2
    ON t1.shipment_id = t2.log_shipment_id;


-- 2. Idle Staff
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.idle_staff;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.idle_staff AS
SELECT t1.*
FROM shipment_temp_raw6 t1
LEFT JOIN logistics_shipment_temp_3 t2
    ON t1.shipment_id = t2.log_shipment_id
WHERE t2.log_shipment_id IS NULL;

In [0]:
%sql
SELECT 
    t1.shipment_id,
    t1.origin_hub_city,
    t2.shipment_id,
    t2.origin_hub_city
FROM shipment_temp_raw6 t1
JOIN shipment_temp_raw6 t2
    ON t1.origin_hub_city = t2.origin_hub_city
WHERE t1.shipment_id != t2.shipment_id;

In [0]:
%sql
-- Drop existing table if it exists
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.invalid_driver_to_shipments;

-- Create a new table capturing shipments with no driver assignment
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.invalid_driver_to_shipments AS
SELECT 
    t2.*   -- Keep all shipment details
FROM shipment_temp_raw6 t1
RIGHT JOIN logistics_shipment_temp_3 t2
    ON t1.shipment_id = t2.log_shipment_id
WHERE t1.shipment_id IS NULL;  -- Only shipments with no matching driver

In [0]:
%sql
SELECT *
FROM shipment_temp_raw6 t1
FULL OUTER JOIN logistics_shipment_temp_3 t2
    ON t1.shipment_id = t2.log_shipment_id
WHERE t1.shipment_id IS NULL 
   OR t2.log_shipment_id IS NULL;

In [0]:
%sql
WITH driver_tl1 AS (
    SELECT *
    FROM shipment_temp_raw6
    WHERE role = 'driver'
),
pending_shipment AS (
    SELECT *
    FROM logistics_shipment_temp_3
    WHERE shipment_status = 'DELAYED'
)
SELECT *
FROM driver_tl1 t1
JOIN pending_shipment t2
ON 1 = 1;

In [0]:
%sql
--left semi join
select * from shipment_temp_raw6 t1 left semi join logistics_shipment_temp_3 t2 on t1.shipment_id=t2.log_shipment_id;
--left anti join
select * from shipment_temp_raw6 t1 left anti join logistics_shipment_temp_3 t2 on t1.shipment_id=t2.log_shipment_id;


In [0]:
%sql
select t1.origin_hub_city from shipment_temp_raw6 t1 left semi join master_city_list t2 on t1.origin_hub_city=t2.city_name;


In [0]:
%sql
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.geo_tagged_staff_data;
CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.geo_tagged_staff_data AS
select t1.*,t2.latitude,t2.longitude from shipment_temp_raw6 t1 left join master_city_list t2 on t1.origin_hub_city=t2.city_name;
   

In [0]:
%sql
drop table if exists usecase_data.logistics_sql_data.wide_shipment_history;
create table if not exists usecase_data.logistics_sql_data.wide_shipment_history as
select * from logistics_shipment_temp_3 t1 left join shipment_temp_raw6 t2 on t1.log_shipment_id=t2.shipment_id left join master_city_list t3 on t2.origin_hub_city=t3.city_name
    

In [0]:
%sql
drop table if exists usecase_data.logistics_sql_data.top_driver_list;
create table if not exists usecase_data.logistics_sql_data.top_driver_list as
with driver_list as (
select * from usecase_data.logistics_sql_data.wide_shipment_history where role='driver')
select * from driver_list qualify row_number() over(partition by origin_hub_city order by shipment_cost desc) <=3


In [0]:
%sql
select *,date_diff(shipment_date,(lag(shipment_date) over(partition by log_shipment_id order by shipment_date asc))) as elapsed_days from logistics_shipment_temp_3 
  

In [0]:
%sql
drop table if exists usecase_data.logistics_sql_data.migrant_staff;
create table if not exists usecase_data.logistics_sql_data.migrant_staff as
select shipment_id from shipment_temp_1 intersect select shipment_id from shipment_temp_2;
drop table if exists usecase_data.logistics_sql_data.new_hires;
create table if not exists usecase_data.logistics_sql_data.new_hires as
select shipment_id from shipment_temp_2 except select shipment_id from shipment_temp_1;

In [0]:
%sql
DROP TABLE IF EXISTS usecase_data.logistics_sql_data.multi_level_report;

CREATE TABLE IF NOT EXISTS usecase_data.logistics_sql_data.multi_level_report AS
SELECT 
    SUM(shipment_cost) AS total_cost, 
    origin_hub_city, 
    vehicle_type
FROM usecase_data.logistics_sql_data.staff_with_shipments
GROUP BY CUBE(origin_hub_city, vehicle_type);

In [0]:
multilevel_subtotal_report = (
    inner_joined_df
    .cube("origin_hub_city", "vehicle_type")   # generate all combinations + subtotals
    .agg(sum("shipment_cost").alias("total_cost"))  # aggregate shipment_cost
)

display(multilevel_subtotal_report)